# Model Assessment and Adjusted R-squared: Complete Guide

## 1. Introduction
- When building Multiple Linear Regression models, we need ways to assess how well the model fits the data.
- The standard Coefficient of Determination (R-squared) is a popular metric, but it contains a fundamental, dangerous flaw when used with multiple predictors.
- In this guide, we will explore why R-squared breaks down, the concept of overfitting, and how Adjusted R-squared solves these issues by penalizing unnecessary complexity.
- We will see how Adjusted R-squared acts as a mathematical implementation of Occam's Razor.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Set display options for better readability
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Use a clean visual style for plots
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)

print("Libraries successfully imported! Environment is ready.")

## 3. Data Creation: Synthetic Car Price Dataset
- To demonstrate model assessment, we need a realistic multivariable dataset.
- We will create a synthetic dataset simulating Used Car Prices.
- The true population model will depend on meaningful features:
  - Age (years)
  - Mileage (thousands of miles)
  - Engine Size (Liters)
- Later, we will deliberately introduce useless "noise" variables to see how our metrics react.

In [ ]:
# Generate meaningful synthetic data
n_samples = 200

# Meaningful features
age = np.random.uniform(1, 15, size=n_samples)
mileage = age * 12 + np.random.normal(0, 15, size=n_samples) # Correlation between age and mileage
mileage = np.maximum(5, mileage) # Ensure no negative mileage
engine_size = np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0, 5.0], size=n_samples)

# True Price Formula (The Hidden Reality)
# Base price 35k - 1k per year - 0.1k per thousand miles + 3k per liter of engine size
true_error = np.random.normal(loc=0, scale=3, size=n_samples)
price_k = 35 - (1.0 * age) - (0.1 * mileage) + (3.0 * engine_size) + true_error

# Combine into a Pandas DataFrame
df_cars = pd.DataFrame({
    'age_years': age,
    'mileage_k': mileage,
    'engine_size_L': engine_size,
    'price_k': price_k
})

print(f"Synthetic car dataset created with {n_samples} records and meaningful features.")

## 4. Initial Data Exploration
- Before fitting models, let's briefly inspect the actual data.
- We will check the first few rows and basic descriptive statistics.

In [ ]:
print("--- First 5 rows of the dataset ---")
print(df_cars.head())
print("\n--- Descriptive Statistics ---")
print(df_cars.describe().round(2))

## 5. Core Concept 1: Revisiting R-squared
- The total variation in the response variable (SST) is decomposed into:
  - SSR: Variation explained by the model
  - SSE: Unexplained variation (residual error)
- SST = SSR + SSE
- R-squared = SSR / SST
- Let's fit a multiple regression model using our meaningful predictors.

In [ ]:
# Fit Base Model with meaningful predictors
base_model = smf.ols('price_k ~ age_years + mileage_k + engine_size_L', data=df_cars).fit()

print("--- Base Model Results ---")
print(f"R-squared: {base_model.rsquared:.4f}")
print("This indicates the proportion of variance in Price explained by Age, Mileage, and Engine Size.")

# We can also calculate it manually to prove the math
y_mean = df_cars['price_k'].mean()
SST = np.sum((df_cars['price_k'] - y_mean)**2)
SSE = np.sum(base_model.resid**2)
manual_r2 = 1 - (SSE / SST)

print(f"Manual R-squared Calculation: {manual_r2:.4f}")

## 6. The Fundamental Flaw of R-squared
- R-squared appears ideal, but it has a massive flaw in multiple regression:
- **R-squared NEVER decreases when predictors are added.**
- Even if we add completely meaningless random noise, R-squared will go up because the optimization algorithm uses the extra degree of freedom to slightly reduce training error.
- Let's prove this by adding 20 useless variables.

In [ ]:
# Create a copy of the dataset to add noise to
df_noise = df_cars.copy()

# Add 20 columns of pure random noise (e.g., driver's lucky number, random coin flips, etc.)
num_noise_vars = 20
noise_columns = []
for i in range(1, num_noise_vars + 1):
    col_name = f'noise_var_{i}'
    df_noise[col_name] = np.random.normal(0, 1, size=n_samples)
    noise_columns.append(col_name)

# Build the formula dynamically
formula_noise = 'price_k ~ age_years + mileage_k + engine_size_L + ' + ' + '.join(noise_columns)

# Fit the new "complex" model
noise_model = smf.ols(formula_noise, data=df_noise).fit()

print("--- The Flaw of R-squared ---")
print(f"Original Base Model R-squared: {base_model.rsquared:.4f}")
print(f"Complex Noise Model R-squared: {noise_model.rsquared:.4f}")
print(f"Notice that R-squared INCREASED by {(noise_model.rsquared - base_model.rsquared):.4f}!")
print("This is a dangerous trap. The model looks better, but it is just learning noise.")

## 7. Concept 2: Overfitting and The Bias-Variance Tradeoff
- When a model learns random sample noise instead of underlying structure, this is called **Overfitting**.
- The model becomes too flexible. It creates a "flexible curve" that perfectly weaves through the training points but fails miserably on unseen data.
- We need a penalty that balances Explanatory Power (fit) against Model Complexity (number of predictors).

In [ ]:
# Let's visualize how adding noise variables ALWAYS increases R-squared
r2_history = [base_model.rsquared]
current_formula = 'price_k ~ age_years + mileage_k + engine_size_L'

# Iteratively add one noise variable at a time and track R-squared
for col in noise_columns:
    current_formula += f' + {col}'
    temp_model = smf.ols(current_formula, data=df_noise).fit()
    r2_history.append(temp_model.rsquared)

plt.figure(figsize=(10, 5))
plt.plot(range(0, num_noise_vars + 1), r2_history, marker='o', color='red', linestyle='-')
plt.title('The Flaw of Ordinary R-squared: It Always Goes Up', fontsize=14)
plt.xlabel('Number of Useless Noise Variables Added', fontsize=12)
plt.ylabel('Ordinary R-squared Value', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("As shown, the metric just keeps climbing, giving a false sense of confidence.")

## 8. Concept 3: The Solution - Adjusted R-squared
- Adjusted R-squared modifies ordinary R-squared by incorporating a complexity penalty.
- Formula:
  Adjusted R-squared = 1 - ( (SSE / (n - k - 1)) / (SST / (n - 1)) )
- Where:
  n = Number of observations
  k = Number of predictor variables
- The core logic: Adding a predictor increases k. This shrinks the denominator (n - k - 1), artificially inflating the penalty unless the new variable significantly reduces SSE.

In [ ]:
# Manual calculation of Adjusted R-squared for the Noise Model
n = len(df_noise)
k_base = 3 # Age, Mileage, Engine Size
k_noise = 3 + num_noise_vars # Base features + 20 noise features

SSE_noise = np.sum(noise_model.resid**2)
SST_total = np.sum((df_noise['price_k'] - y_mean)**2)

# Mean Square Error and Mean Square Total
MSE_noise = SSE_noise / (n - k_noise - 1)
MST_total = SST_total / (n - 1)

manual_adj_r2_noise = 1 - (MSE_noise / MST_total)

print("--- Manual Adjusted R-squared Calculation ---")
print(f"Number of observations (n): {n}")
print(f"Number of predictors (k): {k_noise}")
print(f"Statsmodels Adj R-squared:  {noise_model.rsquared_adj:.4f}")
print(f"Manually Computed Adj R-squared: {manual_adj_r2_noise:.4f}")

## 9. Comparing the Metrics
- Unlike ordinary R-squared, Adjusted R-squared CAN and WILL decrease if a variable is useless.
- It acts as a metric for 'Performance-per-Parameter'.
- Let's look at the metrics side-by-side for our Base Model vs the Complex Noise Model.

In [ ]:
print("--- Metric Comparison Table ---")
print(f"Model               | Ordinary R-squared | Adjusted R-squared")
print(f"--------------------|--------------------|-------------------")
print(f"Base (3 features)   | {base_model.rsquared:.4f}             | {base_model.rsquared_adj:.4f}")
print(f"Noise (23 features) | {noise_model.rsquared:.4f}             | {noise_model.rsquared_adj:.4f}")

print("\n--- Conclusion ---")
print("While Ordinary R-squared PREFERS the Noise model, Adjusted R-squared PREFERS the Base model.")
print("Adjusted R-squared correctly identifies that the extra 20 parameters are not worth the complexity.")

## 10. Concept 4: Can Adjusted R-squared Be Negative?
- Yes!
- If a model performs worse than expected by random chance, Adjusted R-squared drops below zero.
- This indicates that the predictors contribute essentially no useful information, but consume degrees of freedom.
- Let's demonstrate this by predicting Price using ONLY random noise.

In [ ]:
# Fit a model predicting Price using ONLY 5 completely random variables
formula_pure_noise = 'price_k ~ noise_var_1 + noise_var_2 + noise_var_3 + noise_var_4 + noise_var_5'
pure_noise_model = smf.ols(formula_pure_noise, data=df_noise).fit()

print("--- Predicting with Pure Noise ---")
print(f"Ordinary R-squared: {pure_noise_model.rsquared:.4f} (Positive, misleadingly implies some explanation)")
print(f"Adjusted R-squared: {pure_noise_model.rsquared_adj:.4f} (Negative! The brutal truth)")
print("\nA negative Adjusted R-squared is a huge warning signal: The model is functionally useless.")

## 11. Concept 5: Model Selection, Parsimony, and Occam's Razor
- Adjusted R-squared operationalizes **Occam's Razor**: Simpler explanations are preferred unless added complexity produces meaningful gains.
- A model is **Parsimonious** when it explains as much variation as possible using the fewest predictors.
- Let's visualize both R-squared and Adjusted R-squared side-by-side as complexity increases.

In [ ]:
# Track both metrics iteratively as we add all 20 noise variables
adj_r2_history = [base_model.rsquared_adj]
current_formula = 'price_k ~ age_years + mileage_k + engine_size_L'

for col in noise_columns:
    current_formula += f' + {col}'
    temp_model = smf.ols(current_formula, data=df_noise).fit()
    adj_r2_history.append(temp_model.rsquared_adj)

plt.figure(figsize=(10, 6))
plt.plot(range(0, num_noise_vars + 1), r2_history, marker='o', color='red', label='Ordinary R-squared')
plt.plot(range(0, num_noise_vars + 1), adj_r2_history, marker='s', color='blue', label='Adjusted R-squared')

plt.title('Occam\'s Razor in Action: R-squared vs Adjusted R-squared', fontsize=14)
plt.xlabel('Number of Noise Variables Added to Base Model', fontsize=12)
plt.ylabel('Metric Score', fontsize=12)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.axvline(x=0, color='green', linestyle='--', label='Optimal Parsimony')

plt.tight_layout()
plt.show()

## 12. Interpreting the Large Gap Warning Signal
- In the plot above, notice the growing gap between the red line and the blue line.
- If: `R-squared - Adjusted R-squared` is large.
- This suggests:
  1. Too many weak predictors
  2. Severe Overfitting Risk
  3. Inflated model complexity

In [ ]:
# Calculate the gap for our models
base_gap = base_model.rsquared - base_model.rsquared_adj
noise_gap = noise_model.rsquared - noise_model.rsquared_adj

print("--- Analyzing the Metric Gap ---")
print(f"Gap in Base Model (3 useful features): {base_gap:.4f}")
print(f"Gap in Complex Model (3 useful + 20 noise): {noise_gap:.4f}")
print("\nThe complex model exhibits a much larger gap, acting as a clear diagnostic red flag.")

## 13. Practice Exercise 1: Evaluating a New Model
**Scenario:** You receive a new dataset of 150 houses. You fit a model using 10 predictors, and Ordinary R-squared is 0.850. 
However, SSE is 5000 and SST is 33333.33.

**Task:** Write code to calculate the Adjusted R-squared manually to verify if the 10 predictors are justified.

In [ ]:
# --- EXERCISE 1 SOLUTION ---
n_houses = 150
k_preds = 10
sse_houses = 5000
sst_houses = 33333.33

# Calculate MSE and MST
mse_houses = sse_houses / (n_houses - k_preds - 1)
mst_houses = sst_houses / (n_houses - 1)

# Calculate Adjusted R-squared
adj_r2_houses = 1 - (mse_houses / mst_houses)
ord_r2_houses = 1 - (sse_houses / sst_houses)

print(f"Exercise 1 Results:")
print(f"Ordinary R-squared: {ord_r2_houses:.4f}")
print(f"Adjusted R-squared: {adj_r2_houses:.4f}")
print(f"Gap: {(ord_r2_houses - adj_r2_houses):.4f}")
print("The model retains a very high Adjusted R-squared, suggesting the predictors are largely justified.")

## 14. Practice Exercise 2: Simulating Pure Noise
**Task:** Create a DataFrame with 50 rows. Target is 'y' (random standard normal). Predictors are 'x1' through 'x15' (all random standard normal). 
Fit an OLS model and print both R-squared and Adjusted R-squared to observe negative Adjusted R-squared.

In [ ]:
# --- EXERCISE 2 SOLUTION ---
np.random.seed(99) # New seed for variety

# Create data
n_ex = 50
df_ex = pd.DataFrame({'y': np.random.normal(0, 1, n_ex)})

ex_formula_terms = []
for i in range(1, 16):
    df_ex[f'x{i}'] = np.random.normal(0, 1, n_ex)
    ex_formula_terms.append(f'x{i}')
    
ex_formula = 'y ~ ' + ' + '.join(ex_formula_terms)

# Fit model
ex_model = smf.ols(ex_formula, data=df_ex).fit()

print(f"Exercise 2 Results (Pure Noise Simulation):")
print(f"Ordinary R-squared: {ex_model.rsquared:.4f} (looks surprisingly high for pure noise!)")
print(f"Adjusted R-squared: {ex_model.rsquared_adj:.4f} (Correctly negative, exposing the truth)")

## 15. Visualization Gallery: Distribution of Residuals
- One of the ways we can visually assess model quality beyond single numbers is by looking at residual patterns.
- The Base Model (meaningful predictors) will have tight residuals.
- The Pure Noise Model will have wide residuals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# Base Model Residuals
sns.histplot(base_model.resid, kde=True, ax=axes[0], color='teal', bins=15)
axes[0].set_title(f'Base Model Residuals\n(Adj R-squared: {base_model.rsquared_adj:.3f})')
axes[0].set_xlabel('Error Value')

# Pure Noise Model Residuals (from exercise 2)
sns.histplot(ex_model.resid, kde=True, ax=axes[1], color='darkred', bins=15)
axes[1].set_title(f'Pure Noise Model Residuals\n(Adj R-squared: {ex_model.rsquared_adj:.3f})')
axes[1].set_xlabel('Error Value')

plt.tight_layout()
plt.show()

## 16. Summary and Key Takeaways

1. **The Flaw of R-squared**: Ordinary R-squared always increases or stays the same when you add predictors. It is useless for comparing models of different complexities.
2. **The Adjusted R-squared Fix**: It incorporates a complexity penalty based on degrees of freedom (n - k - 1).
3. **Can Go Negative**: If predictors are entirely useless, the penalty will drive Adjusted R-squared below zero.
4. **Performance per Parameter**: A predictor should only remain in the model if it improves the fit enough to justify its complexity cost.
5. **Occam's Razor**: Adjusted R-squared helps identify the most parsimonious model mathematically.
6. **Modern Alternatives**: While Adjusted R-squared is great for classical regression, modern machine learning often relies on Train/Test splits, Cross-Validation, and Regularization (L1/L2) for rigorous model assessment.

In [ ]:
print("Module Completed Successfully.")
print("Remember: Explanations should earn their complexity!")